In [1]:
%pip install google-cloud-aiplatform



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 69.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 713.3/713.3 kB 48.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 73.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 29.9 MB/s  0:00:00
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  7/27 [google-crc32c]  WARNING: The script distro is installed in '/usr/local/python/3.12.1/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━ 11/27 [rsa]  WARNING: The scripts pyrsa-decrypt, pyrsa-encrypt, pyrsa-keygen, pyrsa-priv2pub, pyrsa-sign and pyrsa-verify are installed in '/usr/local/python/3.12.1/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppres

In [5]:
from google.cloud import aiplatform
from vertexai.preview.generative_models import GenerativeModel
import json

PROJECT_ID = "general-project-414402"
REGION = "europe-west1"

PROMPT_TEMPLATE = """
Tu es un expert en data quality et SQL.

Ta mission est de transformer une règle fonctionnelle de qualité de données en
spécification technique JSON.

Contraintes STRICTES :
- Répondre UNIQUEMENT en JSON valide
- Toutes les valeurs doivent être des expressions SQL
- Les valeurs SQL doivent être entre double quotes
- Ne pas ajouter de texte hors JSON

Types de règles possibles :
- completeness
- validity
- uniqueness
- condition_validity
- consistency
- continuity

### Règle fonctionnelle
Code règle : {rule_code}
Type de règle : {rule_type}
Table : {table}
Colonne : {column}
Description : {description}

### Schéma de la table
{schema}

### Sortie attendue (JSON)
Format EXACT :
{{
  "rule_code": "",
  "rule_type": "",
  "table": "",
  "column": "",
  "description": "",
  "filter": "",
  "values": ""
}}

Instructions spécifiques :
- filter : expression SQL pour exclure des lignes du contrôle
- values :
    - completeness : valeurs considérées comme NON remplies (hors NULL)
    - validity : valeurs admissibles sous forme SQL ex: "('A','B')"
"""


aiplatform.init(project=PROJECT_ID, location=REGION)

model = GenerativeModel("gemini-2.5-flash")

def generate_rule_spec(rule):
    prompt = PROMPT_TEMPLATE.format(
        rule_code=rule["rule_code"],
        rule_type=rule["rule_type"],
        table=rule["table"],
        column=rule["column"],
        description=rule["description"],
        schema=rule["schema"]
    )

    response = model.generate_content(
        prompt,
        generation_config={
            "temperature": 0.1,
            "max_output_tokens": 512
        }
    )

    # Sécurisation : parsing strict JSON
    try:
        result = json.loads(response.text)
    except json.JSONDecodeError:
        raise ValueError("Réponse Gemini invalide (JSON attendu)")

    return result


rule_input = {
    "rule_code": "REG0001",
    "rule_type": "completeness",
    "table": "client",
    "column": "Brand",
    "description": "La colonne (Brand) doit être remplie sauf si le client est un particulier",
    "schema": """
    client(
      IdClient INT,
      NomClient STRING,
      Brand STRING,
      TypeClient STRING
    )
    """
}

spec = generate_rule_spec(rule_input)
print(json.dumps(spec, indent=2, ensure_ascii=False))


'''
Expected result

{
  "rule_code": "REG0001",
  "rule_type": "completeness",
  "table": "client",
  "column": "Brand",
  "description": "La colonne (Brand) doit être remplie sauf si le client est un particulier",
  "filter": "TypeClient != \"particulier\"",
  "values": "\"\""
}

'''

ValueError: Réponse Gemini invalide (JSON attendu)